## Exploration of Zebrahub expression data to create a list of highly variable genes.

In [ ]:
import anndata
import os

import importlib.util

# Get the full path to the module
module_path = '/hpc/mydata/mathias.voges/Projects/research/zf-decima/daniodecima-main/src/decima/read_hdf5.py'

# Load the module
spec = importlib.util.spec_from_file_location("read_hdf5", module_path)
read_hdf5 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(read_hdf5)

# src_dir = f'{os.path.dirname(__file__)}/../src/decima/'
# sys.path.append(src_dir)
#from read_hdf5 import HDF5Dataset
# from lightning import LightningModel

In [ ]:

# # Parse arguments
# parser = argparse.ArgumentParser()
# parser.add_argument("--name", type=str)
# parser.add_argument("--dir", type=str)
# parser.add_argument("--lr", type=float)
# parser.add_argument("--weight", type=float)
# parser.add_argument("--grad", type=int)
# parser.add_argument("--replicate", type=int, default=0)
# parser.add_argument("--bs", type=int, default=4)
# args = parser.parse_args()


# Get paths
#data_dir = "/hpc/mydata/mathias.voges/Projects/research/zf-decima/outputs/grelu/decima/"
data_dir = "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/data/celltypes_chrom_split_v1"
matrix_file = os.path.join(data_dir, "zebrahub_aggregated.h5ad")
h5_file = os.path.join(data_dir, "data.h5")
print(f"Data paths: {matrix_file}, {h5_file}")

# Load data
print("Reading anndata")
#print(os.getcwd())
ad_raw = anndata.read_h5ad(matrix_file)
ad_out_human = anndata.read_h5ad("/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136584/0/task_0/lr_3e-05_bs_4_w_0.0001/version_0/data_out_decima_19136584-0_20250529_Human_Borzoi_0.h5ad")
ad_out_random = anndata.read_h5ad("/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136570/0/task_0/lr_1.154578199918017e-06_bs_4_w_0.0001/version_0/data_out_decima_19136570-0_20250529_Random_0.h5ad")

print(ad_raw)
print(ad_out_human)
print(ad_out_random)


# Make datasets
# print("Making dataset objects")
# train_dataset = read_hdf5.GeneForecastDataset(h5_file=h5_file, ad=ad, key="train", max_seq_shift=5000, augment_mode="random", seed=0, history_length=7, forecast_horizon=3)
# val_dataset = read_hdf5.GeneForecastDataset(h5_file=h5_file, ad=ad, key="val", max_seq_shift=0, history_length=7, forecast_horizon=3)

In [ ]:
from matplotlib import pyplot as plt
plt.scatter(ad_out_human.X[:, 0], ad_out_human.layers['scaled'][:, 0])
print(ad_out_human.X[:, 0].mean())
print(ad_out_human.var['mean_counts'][0])

In [ ]:
plt.scatter(ad_out_human.var['mean_counts'], ad_out_human.var['pearson'], alpha=0.01)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Calculate standard deviation
gene_std = ad_out_human.X.std(axis=0)

# Create the plot with better visualization
plt.figure(figsize=(10, 6))
plt.scatter(gene_std, ad_out_human.var['pearson'], alpha=0.1, s=10)  # s=10 for smaller points

# Add a trend line
z = np.polyfit(gene_std, ad_out_human.var['pearson'], 1)
p = np.poly1d(z)
plt.plot(gene_std, p(gene_std), "r--", alpha=0.8)

# Add labels and title
plt.xlabel('Gene Standard Deviation')
plt.ylabel('Pearson Correlation')
plt.title('Relationship between Gene Variance and Pearson Correlation')

# Add correlation coefficient
correlation = np.corrcoef(gene_std, ad_out_human.var['pearson'])[0,1]
plt.text(0.05, 0.95, f'Correlation: {correlation:.3f}', 
         transform=plt.gca().transAxes, 
         bbox=dict(facecolor='white', alpha=0.8))

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Calculate standard deviation
gene_std = ad_out_random.X.std(axis=0)

# Create the plot with better visualization
plt.figure(figsize=(10, 6))
plt.scatter(gene_std, ad_out_random.var['pearson'], alpha=0.1, s=10)  # s=10 for smaller points

# Add a trend line
z = np.polyfit(gene_std, ad_out_random.var['pearson'], 1)
p = np.poly1d(z)
plt.plot(gene_std, p(gene_std), "r--", alpha=0.8)

# Add labels and title
plt.xlabel('Gene Standard Deviation')
plt.ylabel('Pearson Correlation')
plt.title('Relationship between Gene Variance and Pearson Correlation')

# Add correlation coefficient
correlation = np.corrcoef(gene_std, ad_out_random.var['pearson'])[0,1]
plt.text(0.05, 0.95, f'Correlation: {correlation:.3f}', 
         transform=plt.gca().transAxes, 
         bbox=dict(facecolor='white', alpha=0.8))

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Calculate standard deviation (using either dataset since they should have same genes)
gene_std = ad_out_human.X.std(axis=0)

# Calculate difference in Pearson correlation
pearson_diff = ad_out_human.var['pearson'] - ad_out_random.var['pearson']

# Create the plot
plt.figure(figsize=(10, 6))
plt.scatter(gene_std, pearson_diff, alpha=0.1, s=10)

# Add a trend line
z = np.polyfit(gene_std, pearson_diff, 1)
p = np.poly1d(z)
plt.plot(gene_std, p(gene_std), "r--", alpha=0.8)

# Add labels and title
plt.xlabel('Gene Standard Deviation')
plt.ylabel('Pearson Correlation Difference (Human - Random)')
plt.title('Difference in Pearson Correlation vs Gene Standard Deviation')

# Add correlation coefficient
correlation = np.corrcoef(gene_std, pearson_diff)[0,1]
plt.text(0.05, 0.95, f'Correlation: {correlation:.3f}', 
         transform=plt.gca().transAxes, 
         bbox=dict(facecolor='white', alpha=0.8))

# Add horizontal line at y=0 to show where models perform equally
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import median_abs_deviation

def trimmed_std(x, axis=0, proportiontocut=0.1):
    """Calculate trimmed standard deviation"""
    sorted_data = np.sort(x, axis=axis)
    n = x.shape[axis]
    trim = int(n * proportiontocut)
    trimmed_data = sorted_data[trim:-trim] if trim > 0 else sorted_data
    return np.std(trimmed_data, axis=axis)

# Calculate different variance measures (using human data)
gene_std = ad_out_human.X.std(axis=0)
gene_mad = median_abs_deviation(ad_out_human.X, axis=0)
gene_iqr = np.percentile(ad_out_human.X, 75, axis=0) - np.percentile(ad_out_human.X, 25, axis=0)
gene_trimmed_std = trimmed_std(ad_out_human.X, axis=0, proportiontocut=0.1)

# Calculate difference in Pearson correlation
pearson_diff = ad_out_human.var['pearson'] - ad_out_random.var['pearson']

# Create subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot each measure against Pearson difference
measures = {
    'Standard Deviation': gene_std,
    'Median Absolute Deviation': gene_mad,
    'Interquartile Range': gene_iqr,
    'Trimmed Standard Deviation': gene_trimmed_std
}

for (title, measure), ax in zip(measures.items(), axes.ravel()):
    # Scatter plot
    ax.scatter(measure, pearson_diff, alpha=0.1, s=10)
    
    # Add trend line
    z = np.polyfit(measure, pearson_diff, 1)
    p = np.poly1d(z)
    ax.plot(measure, p(measure), "r--", alpha=0.8)
    
    # Add correlation coefficient
    correlation = np.corrcoef(measure, pearson_diff)[0,1]
    ax.text(0.05, 0.95, f'Correlation: {correlation:.3f}', 
            transform=ax.transAxes, 
            bbox=dict(facecolor='white', alpha=0.8))
    
    # Add horizontal line at y=0
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    ax.set_xlabel(title)
    ax.set_ylabel('Pearson Difference (Human - Random)')
    ax.set_title(f'{title} vs Pearson Difference')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import median_abs_deviation

def trimmed_std(x, axis=0, proportiontocut=0.1):
    """Calculate trimmed standard deviation"""
    sorted_data = np.sort(x, axis=axis)
    n = x.shape[axis]
    trim = int(n * proportiontocut)
    trimmed_data = sorted_data[trim:-trim] if trim > 0 else sorted_data
    return np.std(trimmed_data, axis=axis)

# Calculate different variance measures (using human data)
gene_std = ad_out_human.X.std(axis=0)
gene_mad = median_abs_deviation(ad_out_human.X, axis=0)
gene_iqr = np.percentile(ad_out_human.X, 75, axis=0) - np.percentile(ad_out_human.X, 25, axis=0)
gene_trimmed_std = trimmed_std(ad_out_human.X, axis=0, proportiontocut=0.1)

# Calculate mean expression
mean_expr = ad_out_human.X.mean(axis=0)

# Calculate difference in Pearson correlation
pearson_diff = ad_out_human.var['pearson'] - ad_out_random.var['pearson']

# Apply filters
# 1. Mean expression filter
min_mean = 0.1
mean_filter = mean_expr > min_mean

# 2. Variance filters
min_variance = np.percentile(gene_std, 10)  # keep top 90% of genes by variance
variance_filter = gene_std > min_variance

# 3. CV filter
cv = gene_std / mean_expr
min_cv = 0.1
cv_filter = cv > min_cv

# Combine filters
high_variance_genes = mean_filter & variance_filter & cv_filter

print(f"Total genes: {len(gene_std)}")
print(f"Genes passing mean filter: {mean_filter.sum()}")
print(f"Genes passing variance filter: {variance_filter.sum()}")
print(f"Genes passing CV filter: {cv_filter.sum()}")
print(f"Final number of high-variance genes: {high_variance_genes.sum()}")

# Create subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot each measure against Pearson difference (only for filtered genes)
measures = {
    'Standard Deviation': gene_std,
    'Median Absolute Deviation': gene_mad,
    'Interquartile Range': gene_iqr,
    'Trimmed Standard Deviation': gene_trimmed_std
}

for (title, measure), ax in zip(measures.items(), axes.ravel()):
    # Scatter plot (only filtered genes)
    ax.scatter(measure[high_variance_genes], 
              pearson_diff[high_variance_genes], 
              alpha=0.1, s=10)
    
    # Add trend line
    z = np.polyfit(measure[high_variance_genes], 
                  pearson_diff[high_variance_genes], 1)
    p = np.poly1d(z)
    ax.plot(measure[high_variance_genes], 
            p(measure[high_variance_genes]), 
            "r--", alpha=0.8)
    
    # Add correlation coefficient
    correlation = np.corrcoef(measure[high_variance_genes], 
                            pearson_diff[high_variance_genes])[0,1]
    ax.text(0.05, 0.95, f'Correlation: {correlation:.3f}', 
            transform=ax.transAxes, 
            bbox=dict(facecolor='white', alpha=0.8))
    
    # Add horizontal line at y=0
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    ax.set_xlabel(title)
    ax.set_ylabel('Pearson Difference (Human - Random)')
    ax.set_title(f'{title} vs Pearson Difference\n(Filtered Genes)')

plt.tight_layout()
plt.show()

# Optional: Create a summary plot of the filtering
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(mean_expr, gene_std, alpha=0.1, s=10, label='All genes')
ax.scatter(mean_expr[high_variance_genes], 
          gene_std[high_variance_genes], 
          alpha=0.5, s=10, label='High variance genes')
ax.set_xlabel('Mean Expression')
ax.set_ylabel('Standard Deviation')
ax.set_title('Gene Filtering Summary')
ax.legend()
plt.show()

## Caclulate and save highly variable genes used in later analysis. 

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import median_abs_deviation

def trimmed_std(x, axis=0, proportiontocut=0.1):
    """Calculate trimmed standard deviation"""
    sorted_data = np.sort(x, axis=0)
    n = x.shape[axis]
    trim = int(n * proportiontocut)
    trimmed_data = sorted_data[trim:-trim] if trim > 0 else sorted_data
    return np.std(trimmed_data, axis=axis)

# Make a copy of the data to avoid modifying the original
ad_human = ad_out_human.copy()

# Calculate mean and variance
mean = ad_human.X.mean(axis=0)
var = ad_human.X.var(axis=0)

# Calculate dispersion (variance/mean)
dispersion = var / mean

# Add these to var
ad_human.var['mean'] = mean
ad_human.var['dispersions'] = dispersion

# Run scanpy's highly variable genes detection
sc.pp.highly_variable_genes(ad_human, 
                          min_mean=0.0125, 
                          max_mean=5, 
                          min_disp=0.5)

# Get the highly variable genes mask
high_variance_genes = ad_human.var['highly_variable']

print(f"Total genes: {ad_human.n_vars}")
print(f"Number of highly variable genes: {high_variance_genes.sum()}")

# Calculate different variance measures (using human data)
gene_std = ad_out_human.X.std(axis=0)
gene_mad = median_abs_deviation(ad_out_human.X, axis=0)
gene_iqr = np.percentile(ad_out_human.X, 75, axis=0) - np.percentile(ad_out_human.X, 25, axis=0)
gene_trimmed_std = trimmed_std(ad_out_human.X, axis=0, proportiontocut=0.1)

# Calculate difference in Pearson correlation
pearson_diff = ad_out_human.var['pearson'] - ad_out_random.var['pearson']

# Create subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot each measure against Pearson difference (only for filtered genes)
measures = {
    'Standard Deviation': gene_std,
    'Median Absolute Deviation': gene_mad,
    'Interquartile Range': gene_iqr,
    'Trimmed Standard Deviation': gene_trimmed_std
}

for (title, measure), ax in zip(measures.items(), axes.ravel()):
    # Scatter plot (only filtered genes)
    ax.scatter(measure[high_variance_genes], 
              pearson_diff[high_variance_genes], 
              alpha=0.1, s=10)
    
    # Add trend line
    z = np.polyfit(measure[high_variance_genes], 
                  pearson_diff[high_variance_genes], 1)
    p = np.poly1d(z)
    ax.plot(measure[high_variance_genes], 
            p(measure[high_variance_genes]), 
            "r--", alpha=0.8)
    
    # Add correlation coefficient
    correlation = np.corrcoef(measure[high_variance_genes], 
                            pearson_diff[high_variance_genes])[0,1]
    ax.text(0.05, 0.95, f'Correlation: {correlation:.3f}', 
            transform=ax.transAxes, 
            bbox=dict(facecolor='white', alpha=0.8))
    
    # Add horizontal line at y=0
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    ax.set_xlabel(title)
    ax.set_ylabel('Pearson Difference (Human - Random)')
    ax.set_title(f'{title} vs Pearson Difference\n(Highly Variable Genes)')

plt.tight_layout()
plt.show()

# Create a summary plot of the filtering
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(ad_human.var['mean'], 
          ad_human.var['dispersions'], 
          alpha=0.1, s=10, label='All genes')
ax.scatter(ad_human.var['mean'][high_variance_genes], 
          ad_human.var['dispersions'][high_variance_genes], 
          alpha=0.5, s=10, label='Highly variable genes')
ax.set_xlabel('Mean Expression')
ax.set_ylabel('Dispersion')
ax.set_title('Gene Filtering Summary (scanpy HVG)')
ax.legend()
plt.show()

In [ ]:
# ... existing code ...

# Define HVGs and LVGs
hvg_mask = ad_human.var['highly_variable']
lvg_mask = ~ad_human.var['highly_variable']

# Get mean expression for each gene
mean_expression = ad_human.var['mean']

# Optionally, sample a subset for plotting if there are too many genes
n_genes_to_plot = 500  # adjust as needed
hvg_means = mean_expression[hvg_mask]
lvg_means = mean_expression[lvg_mask]

if len(hvg_means) > n_genes_to_plot:
    hvg_means = hvg_means.sample(n_genes_to_plot, random_state=42)
if len(lvg_means) > n_genes_to_plot:
    lvg_means = lvg_means.sample(n_genes_to_plot, random_state=42)

# Plot histograms
plt.figure(figsize=(12, 6))
plt.hist(lvg_means, bins=30, alpha=0.6, label='Low Variable Genes', color='blue', density=True)
plt.hist(hvg_means, bins=30, alpha=0.6, label='Highly Variable Genes', color='orange', density=True)
plt.xlabel('Mean Expression')
plt.ylabel('Density')
plt.title('Expression Histogram: Highly Variable vs Low Variable Genes')
plt.legend()
plt.show()

In [ ]:
# Pick random genes from each group
n_genes = 10
hvg_genes = ad_human.var.index[hvg_mask].to_series().sample(n_genes, random_state=42)
lvg_genes = ad_human.var.index[lvg_mask].to_series().sample(n_genes, random_state=42)

# Plot expression histograms for each group
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for gene in hvg_genes:
    axes[0].hist(ad_human[:, gene].X.flatten(), bins=30, alpha=0.3, color='orange')
axes[0].set_title('Expression Distribution: Random HVGs')
axes[0].set_ylabel('Frequency')

for gene in lvg_genes:
    axes[1].hist(ad_human[:, gene].X.flatten(), bins=30, alpha=0.3, color='blue')
axes[1].set_title('Expression Distribution: Random LVGs')
axes[1].set_xlabel('Expression')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# 1. Extract HVG list
hvg_list = ad_human.var.index[ad_human.var['highly_variable']].tolist()

# 2. Subset your DataFrame
df_hvg = df[df['index'].isin(hvg_list)]

# 3. Now df_hvg contains only rows for highly variable genes
print(df_hvg.head())

In [ ]:
# Save to a text file
with open("highly_variable_genes.txt", "w") as f:
    for gene in hvg_list:
        f.write(f"{gene}\n")